## Ingest results.json


In [0]:
%run "../utils/config"

### Step 1 - Read the json file


#### 1.1 Define the schema


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType,  StringType


In [0]:
results_schema = StructType([
    StructField("resultId", IntegerType(), False),
    StructField("raceId", IntegerType(), True),
    StructField("driverId", IntegerType(), True),
    StructField("constructorId", IntegerType(), True),
    StructField("number", IntegerType(), True),
    StructField("grid", IntegerType(), True),
    StructField("position", IntegerType(), True),
    StructField("positionText", StringType(), True),
    StructField("positionOrder", IntegerType(), True),
    StructField("points", FloatType(), True),
    StructField("laps", IntegerType(), True),
    StructField("time", StringType(), True),
    StructField("milliseconds", IntegerType(), True),
    StructField("fastestLap", IntegerType(), True),
    StructField("rank", IntegerType(), True),
    StructField("fastestLapTime", StringType(), True),
    StructField("fastestLapSpeed", FloatType(), True),
    StructField("statusId", StringType(), True)
])


#### 1.2 Read the json file


In [0]:
raw_path = raw_folder_path


In [0]:
results_df = (
    spark.read
        .format("json")
        .schema(results_schema)
        .load(f"{raw_path}/results.json")
    )


### Step 2 - Transform the data


In [0]:
results_renamed_df = (
    results_df
        .withColumnRenamed("resultId", "result_id")
        .withColumnRenamed("raceId", "race_id")
        .withColumnRenamed("driverId", "driver_id")
        .withColumnRenamed("constructorId", "constructor_id")
        .withColumnRenamed("positionText", "position_text")
        .withColumnRenamed("positionOrder", "position_order")
        .withColumnRenamed("fastestLapTime", "fastest_lap_time")
        .withColumnRenamed("fastestLapSpeed", "fastest_lap_speed")
)


In [0]:
from pyspark.sql.functions import col, current_timestamp


In [0]:
results_dropped_df = results_renamed_df.drop(col("statusId"))


In [0]:
results_final_df = add_ingestion_date(results_dropped_df)


### Step 3 - Write the output to parquet


In [0]:
processed_path = processed_folder_path


In [0]:
results_final_df.write.format("parquet").mode("overwrite").save(f"{processed_path}/results")
